In [2]:
import pandas as pd
from pathlib import Path

Data From CENTER OF AMERICAN WOMEN AND POLITICS (CAWP) - RUTGERS UNIVERSITY

Women congress: https://cawp.rutgers.edu/candidates-2026-election-results-tracker/women-120th-congress

Current women: https://cawp.rutgers.edu/data/women-elected-officials-database?tab=ByParty

Candidates: https://cawp.rutgers.edu/data/candidates-women-candidates-2026?tab=CandidateList

Rebound candidates: https://cawp.rutgers.edu/data/candidates-women-candidates-2026?tab=ReboundCandidates

Senate 2022: https://dataverse.harvard.edu/file.xhtml?fileId=7412054&version=3.0

women_candidates_2024: https://cawp.rutgers.edu/data/candidates-election-results-and-analysis-0/2024-summary-women-candidates

1974-2024 House: https://dataverse.harvard.edu/ 

Column	Meaning	Example Women congress
State	Two-letter abbreviation of the U.S. state where the candidate is running.	CA = California
2026 Office	Political office the candidate is running for in 2026.	U.S. Rep. = U.S. House of Representatives
2026 Dist.	Congressional or electoral district in which the candidate is running.	2.0 = District 2
First Name	Candidate's first name.	Rose
Middle Name	Candidate's middle name, if available.	Penelope
Last Name	Candidate's surname.	Yee
Party	Candidate's political party.	D = Democratic Party
2026 Seat	Whether the candidate is challenging or defending a seat, or running for an open seat.	C = Challenger
Primary Date	Appears to indicate the primary election outcome or status, rather than an actual date.	W = Won, L = Lost
2024 Office	Office the candidate ran for in the 2024 election.	U.S. Rep.
2024 District	Electoral district associated with the candidate's 2024 race.	1 = District 1
Nominee in 2024	Whether the candidate was their party's nominee in 2024.	Y = Yes, N = No

Important abbreviations
2026 Office
- U.S. Sen. = United States Senator
- U.S. Rep. = United States Representative
- Governor = State Governor
- Sec. St. = Secretary of State
Party
- D = Democratic Party
- R = Republican Party
- I = Independent (if present)
2026 Seat
- C = Challenger (likely)
- O = Open seat (likely)
- I = Incumbent (if present)

In [ ]:
historical_representatives

In [11]:
data_folder = Path("../data").resolve()

In [22]:
women_versus_women = pd.read_csv(data_folder / '2026 Woman V Woman.csv')
women_versus_women[["State", "Office"]] = women_versus_women[["State", "Office"]].ffill()

In [24]:
women_rebound = pd.read_csv(data_folder / '2026 Rebound Candidates.csv')

In [26]:
current_women = pd.read_csv(data_folder / 'search_officeholders-05112026.csv')

In [32]:
women_congress = pd.read_excel(data_folder / '2026 Women congress.xlsx', sheet_name='Data')

In [35]:
senate_2022 = pd.read_csv(data_folder / 'senate_2022.csv')

In [64]:
historical_representatives = pd.read_csv(data_folder / '1976-2024-house.tab', sep=',', dtype={'runoff': 'string'})
historical_representatives['candidate'] = (historical_representatives['candidate'].str.replace(r'\"', '"', regex=False))

In [67]:
representatives_2024 = historical_representatives[historical_representatives['year'] == 2024]

In [ ]:
representatives_2024_clean = representatives_2024[(~representatives_2024['writein'].fillna(False))&~(representatives_2024['candidate'].str.upper()=='WRITEIN')]


representatives_2024_clean_win = (4.sort_values('candidatevotes', ascending=False)
    representatives_2024_clean'], keep='first')
    .loc[representatives_2024_clean['party'].notna()         & representatives_2024_clean['totalvotes'].gt(0)]
    .sort_values('candidatevotes', ascending=False)
    .drop_duplicates(subset=['state', 'district'], keep='first')
)

In [136]:
df_2024=representatives_2024_clean_win.drop_duplicates(subset=['office', 'state_po', 'candidate', 'party']).groupby(['office', 'state_po','party']).size().reset_index(name='Total_2024')
df_2024['office'] = df_2024['office'].replace({'US HOUSE': 'U.S. Representative'})
df_2024['party'] = df_2024['party'].replace({'REPUBLICAN': 'R', 'DEMOCRAT': 'D'})

In [88]:
women_candidates_2024 = pd.read_excel(data_folder / 'women_candidates_2024.xlsx', sheet_name='Data')
women_candidates_2024 = women_candidates_2024[women_candidates_2024['Office'].isin(['U.S. Rep.', 'U.S. Sen.'])]
women_candidates_2024 = women_candidates_2024[women_candidates_2024['General Election Result']=='W']

In [138]:
df_w_2026=women_congress.drop_duplicates(subset=[ 'Office', 'State','First Name', 'Last Name', 'Party']).groupby(['Office','State', 'Party']).size().reset_index(name='Num_Women')


In [139]:
df_w_2024=women_candidates_2024.drop_duplicates(subset=[ 'Office', 'State','First Name', 'Last Name', 'Party']).groupby(['Office','State', 'Party']).size().reset_index(name='Num_Women')
df_w_2024['Office'] = df_w_2024['Office'].replace({'U.S. Representative': 'House', 'U.S. Senator': 'Senate'})
df_w_2024_complete = df_2024.merge(df_w_2024, on=['Office', 'State', 'Party'], how='left', suffixes=('_total', '_women'))


In [140]:
df_w_2026.merge(df_w_2024, on=['Office', 'State', 'Party'], how='outer', suffixes=('_2026', '_2024'))

,Office,State,Party,Num_Women_2026,Num_Women_2024
0,U.S. Rep.,AL,D,NaN,1.0
1,U.S. Rep.,AZ,D,NaN,1.0
2,U.S. Rep.,CA,D,NaN,14.0
3,U.S. Rep.,CA,R,NaN,1.0
4,U.S. Rep.,CO,D,NaN,2.0
...,...,...,...,...,...
125,U.S. Senator,SC,D,1.0,NaN
126,U.S. Senator,WA,D,2.0,NaN
127,U.S. Senator,WI,D,1.0,NaN
128,U.S. Senator,WV,R,1.0,NaN
